# 03 Rung 2: frozen XLS-R-300M, layer sweep (train -> dev)

Embeddings come from `deepvoice.xlsr.extract_cached` (mean-pooled every 3rd layer, fp16, cached in
`data/features/`). For each layer: standardized logistic regression (C=1, class-balanced, seed 0),
trained on train and scored on dev with a speaker-clustered 95% CI. Results: `results/rung2.csv`.
The layer for rung 2 is chosen on dev.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.models import run
from deepvoice.xlsr import LAYERS

p = pd.read_parquet("data/protocols.parquet")
COLS = [f"x{i:04d}" for i in range(1024)]
D = {}
for s in ["train", "dev"]:
    m = p[p.split == s][["file", "label", "speaker"]].reset_index(drop=True)
    assert Path(f"data/features/xlsr300m_{s}_files.txt").read_text().split() == list(m.file)
    D[s] = (m, np.load(f"data/features/xlsr300m_{s}.npy", mmap_mode="r"))

In [ ]:
out = ROOT / "results/rung2.csv"
done = set(pd.read_csv(out).config) if out.exists() else set()
for k, layer in enumerate(LAYERS):
    if f"rung2_logreg_xlsr300m_L{layer:02d}_s0" in done:
        continue
    fr = {s: pd.concat([m, pd.DataFrame(np.asarray(e[:, k], dtype=np.float32), columns=COLS)], axis=1)
          for s, (m, e) in D.items()}
    run(2, "logreg", COLS, fr["train"], [("asvspoof", "train", fr["train"]), ("asvspoof", "dev", fr["dev"])],
        params={"C": 1.0}, feature_set=f"xlsr300m_L{layer:02d}")
r = pd.read_csv(out)
r["layer"] = r.model.str.extract(r"L(\d+)").astype(int)[0]
dev = r[r.split == "dev"].sort_values("layer")
dev[["layer", "eer", "ci_low", "ci_high", "train_time"]].assign(**{c: dev[c] * 100 for c in ["eer", "ci_low", "ci_high"]}).round(3)

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.fill_between(dev.layer, 100 * dev.ci_low, 100 * dev.ci_high, color="#1baf7a", alpha=0.18, lw=0)
ax.plot(dev.layer, 100 * dev.eer, marker="o", ms=6, lw=2, color="#1baf7a", label="XLS-R layer + logistic regression")
ax.axhline(0.49, color=MUTED, ls="--", lw=1)
ax.text(24.3, 0.49, "best classical\n(LFCC40 RF, untuned)", va="center", fontsize=8, color=MUTED)
ax.set_xticks(LAYERS); ax.set_xlim(-0.5, 30)
ax.set_xlabel("XLS-R layer (0 = CNN output, 24 = last)"); ax.set_ylabel("dev EER % (band = 95% CI)")
ax.grid(axis="y", color=GRID, linewidth=0.6)
ax.set_title("Frozen XLS-R-300M: dev EER by layer", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/xlsr_layers.png", dpi=200)
fig.savefig(ROOT / "results/figures/xlsr_layers.pdf")

## Small MLP on the best layers, and the rung 2 choice (decided before eval)

Rule, fixed before any eval run: take the rung 2 candidate with the lowest dev EER; every candidate
whose dev EER lies inside that model's 95% CI counts as tied, and ties are broken by speaker-grouped
5-fold CV EER on train; if CV ties too, the lower dev EER wins.

*Note (20 Nov):* an earlier version broke an exact CV tie by row order; in one re-run that picked
layer 6 and scored it on eval once (0.34% EER, kept in `results/rung2.csv` for transparency). The
final model stays layer 3, the lowest-dev-EER model as `hypotheses.md` defines it. Candidates: logistic regression on each layer, and an MLP (one hidden layer of
256, early stopping) on the tied layers.

In [ ]:
from deepvoice.models import grouped_cv_eer


def frames(layer, splits):
    k = LAYERS.index(layer)
    return {s: pd.concat([D[s][0], pd.DataFrame(np.asarray(D[s][1][:, k], dtype=np.float32), columns=COLS)], axis=1)
            for s in splits}

r = pd.read_csv(out); dev = r[r.split == "dev"].copy()
best = dev.loc[dev.eer.idxmin()]
tied_layers = sorted(int(m.split("L")[1]) for m in dev[dev.eer <= best.ci_high].model)
print("best:", best.model, f"{100 * best.eer:.4f}%", "| tied layers:", tied_layers)
MLP = {"hidden_layer_sizes": (256,), "alpha": 1e-3}
for layer in tied_layers:
    if f"rung2_mlp_xlsr300m_L{layer:02d}_s0" not in set(r.config):
        fr = frames(layer, ["train", "dev"])
        run(2, "mlp", COLS, fr["train"], [("asvspoof", "train", fr["train"]), ("asvspoof", "dev", fr["dev"])],
            params=MLP, feature_set=f"xlsr300m_L{layer:02d}")
r = pd.read_csv(out); dev = r[r.split == "dev"].copy()
best = dev.loc[dev.eer.idxmin()]
tied = dev[dev.eer <= best.ci_high].copy()
for i, row in tied.iterrows():
    name, layer = row.model.split(":")[0], int(row.model.split("L")[1])
    params = {"C": 1.0} if name == "logreg" else MLP
    tied.loc[i, "cv_eer"] = grouped_cv_eer(name, params, frames(layer, ["train"])["train"], COLS)
# ties on CV fall back to the lower dev EER, so the choice never depends on row order
choice = tied.sort_values(["cv_eer", "eer"]).iloc[0]
tied[["model", "eer", "ci_low", "ci_high", "cv_eer"]].assign(**{c: tied[c] * 100 for c in ["eer", "ci_low", "ci_high", "cv_eer"]}).round(4)

In [ ]:
CHOICE = choice.model
print("rung 2 model (chosen on dev + grouped CV, before eval):", CHOICE)

## Rung 2 on ASVspoof eval (scored once)

In [ ]:
from deepvoice.data import REPEATED_ATTACKS
from deepvoice.metrics import per_attack_eer

name, layer = CHOICE.split(":")[0], int(CHOICE.split("L")[1])
params = {"C": 1.0} if name == "logreg" else MLP
m = p[p.split == "eval"][["file", "label", "speaker", "attack"]].reset_index(drop=True)
assert Path("data/features/xlsr300m_eval_files.txt").read_text().split() == list(m.file)
ev = np.load("data/features/xlsr300m_eval.npy", mmap_mode="r")
D["eval"] = (m[["file", "label", "speaker"]], ev)
fr = frames(layer, ["train", "dev", "eval"])
model, rows = run(2, name, COLS, fr["train"], [("asvspoof", s, fr[s]) for s in ["train", "dev", "eval"]],
                  params=params, feature_set=f"xlsr300m_L{layer:02d}")
s = model.predict_proba(fr["eval"][COLS].to_numpy())[:, 1]
pa = pd.Series(per_attack_eer(m.label, s, m.attack)).mul(100)
pa = pd.DataFrame({CHOICE: pa, "type": ["repeated" if a in REPEATED_ATTACKS else "unseen" for a in pa.index]})
pa.to_csv(ROOT / "results/rung2_eval_per_attack.csv", float_format="%.2f")
print({r_["split"]: f"{100*r_['eer']:.2f}% [{100*r_['ci_low']:.2f}, {100*r_['ci_high']:.2f}]" for r_ in rows})
pa.round(2).T